# Baseline Model Performance & Empirical Benchmark

Evaluates **Scikit-Learn Exact Random Forest**, **Extra-Trees**, **LightGBM (GOSS)**, and **FastForest MABSplit Baseline** across 5 dataset scales.

In [ ]:
import os
import sys

# 1. Clone FastForest repository
if not os.path.exists('FastForest'):
    !git clone https://github.com/ThrunGroup/FastForest.git

# 2. Add FastForest to Python Path
sys.path.append(os.path.abspath('FastForest'))

# 3. Install required benchmark packages
!pip install -q numba openml scikit-learn lightgbm psutil matplotlib seaborn pandas numpy

print("Environment setup complete. FastForest repo imported successfully!")

In [ ]:
# ==============================================================================
# CELL 2: Automated Dataset Loader & Preprocessing
# ==============================================================================
import openml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.datasets import make_classification

def preprocess_df(df):
    df_clean = df.copy()
    num_cols = df_clean.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

    cat_cols = df_clean.select_dtypes(include=['category', 'object']).columns
    for col in cat_cols:
        df_clean[col] = LabelEncoder().fit_transform(df_clean[col].astype(str))

    return df_clean

def get_benchmark_datasets():
    datasets = {}

    print("1. Preparing Synthetic Scaling Suite...")
    for N in [10000, 50000, 100000]:
        X_syn, y_syn = make_classification(
            n_samples=N, n_features=20, n_informative=15, n_classes=2, random_state=42
        )
        X_tr, X_te, y_tr, y_te = train_test_split(X_syn, y_syn, test_size=0.2, random_state=42)
        datasets[f'Synthetic_{N}'] = (X_tr, X_te, y_tr, y_te)

    print("2. Fetching Adult Census dataset from OpenML...")
    try:
        adult = openml.datasets.get_dataset(1590)
        X_ad, y_ad, _, _ = adult.get_data(target=adult.default_target_attribute)
        X_ad_clean = preprocess_df(X_ad)
        X_ad_scaled = StandardScaler().fit_transform(X_ad_clean)
        y_ad_enc = LabelEncoder().fit_transform(y_ad)
        X_tr, X_te, y_tr, y_te = train_test_split(X_ad_scaled, y_ad_enc, test_size=0.2, random_state=42, stratify=y_ad_enc)
        datasets['Adult'] = (X_tr, X_te, y_tr, y_te)
    except Exception as e:
        print(f"Adult dataset fetch note: {e}")

    print("3. Fetching Covertype dataset from OpenML...")
    try:
        cov = openml.datasets.get_dataset(150)
        X_cov, y_cov, _, _ = cov.get_data(target=cov.default_target_attribute)
        X_cov_clean = preprocess_df(X_cov)
        X_cov_scaled = StandardScaler().fit_transform(X_cov_clean)
        y_cov_enc = LabelEncoder().fit_transform(y_cov)
        X_cov_sub = X_cov_scaled[:50000]
        y_cov_sub = y_cov_enc[:50000]
        X_tr, X_te, y_tr, y_te = train_test_split(X_cov_sub, y_cov_sub, test_size=0.2, random_state=42, stratify=y_cov_sub)
        datasets['Covertype_50K'] = (X_tr, X_te, y_tr, y_te)
    except Exception as e:
        print(f"Covertype dataset fetch note: {e}")

    print("All benchmark datasets loaded successfully!")
    return datasets

benchmark_datasets = get_benchmark_datasets()

In [ ]:
# ==============================================================================
# CELL 3: Complete & Verified Multi-Seed Baseline Execution Loop
# ==============================================================================
import time
import psutil
import gc
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score

from data_structures.forest_classifier import ForestClassifier as FastForestMAB

SEEDS = [42, 123, 777]
N_TREES = 10
MAX_DEPTH = 10

def get_peak_memory_mb():
    return psutil.Process().memory_info().rss / (1024 * 1024)

results_list = []

for dname, (X_tr, X_te, y_tr, y_te) in benchmark_datasets.items():
    sklearn_models = {
        'Scikit-Learn Exact RF': RandomForestClassifier(n_estimators=N_TREES, max_depth=MAX_DEPTH, random_state=42, n_jobs=4),
        'Extra-Trees': ExtraTreesClassifier(n_estimators=N_TREES, max_depth=MAX_DEPTH, random_state=42, n_jobs=4),
        'LightGBM (GOSS)': LGBMClassifier(n_estimators=N_TREES, max_depth=MAX_DEPTH, random_state=42, verbose=-1),
    }

    for mname, model in sklearn_models.items():
        times, f1s, accs, mems = [], [], [], []
        for seed in SEEDS:
            gc.collect()
            mem_start = get_peak_memory_mb()
            t0 = time.perf_counter()
            if hasattr(model, 'random_state'):
                model.random_state = seed
            model.fit(X_tr, y_tr)
            t1 = time.perf_counter()
            y_pred = model.predict(X_te)
            acc = accuracy_score(y_te, y_pred)
            f1 = f1_score(y_te, y_pred, average='macro')
            mem_peak = get_peak_memory_mb() - mem_start
            times.append(t1 - t0)
            f1s.append(f1)
            accs.append(acc)
            mems.append(max(0, mem_peak))

        res = {
            'Dataset': dname, 'Model': mname,
            'Wall_Time_Mean': np.mean(times), 'Wall_Time_Std': np.std(times),
            'F1_Score_Mean': np.mean(f1s), 'F1_Score_Std': np.std(f1s),
            'Accuracy_Mean': np.mean(accs), 'Accuracy_Std': np.std(accs),
            'RAM_MB_Peak': np.mean(mems)
        }
        results_list.append(res)

    try:
        mname = 'FastForest MABSplit'
        times, f1s, accs, mems = [], [], [], []
        for seed in SEEDS:
            gc.collect()
            mem_start = get_peak_memory_mb()
            t0 = time.perf_counter()
            mab_model = FastForestMAB(data=X_tr, labels=y_tr, n_estimators=N_TREES, max_depth=MAX_DEPTH, solver='MAB', random_state=seed)
            mab_model.fit()
            t1 = time.perf_counter()
            y_pred, _ = mab_model.predict_batch(X_te)
            acc = accuracy_score(y_te, y_pred)
            f1 = f1_score(y_te, y_pred, average='macro')
            mem_peak = get_peak_memory_mb() - mem_start
            times.append(t1 - t0)
            f1s.append(f1)
            accs.append(acc)
            mems.append(max(0, mem_peak))

        res = {
            'Dataset': dname, 'Model': mname,
            'Wall_Time_Mean': np.mean(times), 'Wall_Time_Std': np.std(times),
            'F1_Score_Mean': np.mean(f1s), 'F1_Score_Std': np.std(f1s),
            'Accuracy_Mean': np.mean(accs), 'Accuracy_Std': np.std(accs),
            'RAM_MB_Peak': np.mean(mems)
        }
        results_list.append(res)
    except Exception as e:
        print(f"  [FastForest MABSplit] Note: {e}")

df_results = pd.DataFrame(results_list)
df_results.to_csv('baseline_benchmark_results.csv', index=False)

## HYBRID MABSplit Evaluation

Compares Baseline MABSplit vs **Accelerated FastMAB (FinalYear)** with Hybrid Depth Switch, Numba SIMD Kernels, Coarse-to-Fine Pruning, and Adaptive Minibatch Allocation.

In [ ]:
# ==============================================================================
# CELL 4: Setup Environment for Hybrid MABSplit Comparison
# ==============================================================================
import sys
import os

sys.path.append(os.path.abspath('FinalYear'))
sys.path.append(os.path.abspath('FinalYear/src'))
sys.path.append(os.path.abspath('FastForest/src'))

print("✅ Environment configured for Hybrid MABSplit evaluation!")

In [ ]:
# ==============================================================================
# CELL 5: Import Algorithms & Data Generator
# ==============================================================================
import time
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier

try:
    from data_structures.forest_classifier import ForestClassifier as FastForestMAB
    print("✅ Baseline FastForest MABSplit imported successfully!")
except Exception as e:
    print(f"⚠️ Baseline FastForest import error: {e}")

try:
    from src.models.forest import MABRandomForestClassifier as OurFastMAB
    print("✅ OUR Accelerated FastMAB (FinalYear) imported successfully!")
except Exception as e:
    try:
        from models.forest import MABRandomForestClassifier as OurFastMAB
        print("✅ OUR Accelerated FastMAB (FinalYear) imported successfully!")
    except Exception as e2:
        print(f"⚠️ OUR FastMAB import error: {e2}")

def generate_synthetic_dataset(n_samples=2000, n_features=20, n_informative=15, random_state=42):
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        n_redundant=5,
        n_classes=2,
        random_state=random_state
    )
    return train_test_split(X, y, test_size=0.2, random_state=random_state)

print("✅ Helper functions ready.")

In [ ]:
# ==============================================================================
# CELL 6: Benchmark Models on Small Datasets
# ==============================================================================
SEEDS = [42, 123, 777]
DATASET_SIZES = [2000, 10000]

results_list = []

print("🚀 Starting Small Dataset Baseline & OUR FastMAB Benchmarking across seeds:", SEEDS)
print("=" * 85)

for n_samples in DATASET_SIZES:
    print(f"\n---> Benchmarking Dataset: Synthetic_{n_samples} (Train: {int(n_samples*0.8)} samples, Test: {int(n_samples*0.2)} samples)")

    for seed in SEEDS:
        X_train, X_test, y_train, y_test = generate_synthetic_dataset(n_samples=n_samples, random_state=seed)

        # --- 1. Scikit-Learn Exact RF ---
        rf = RandomForestClassifier(n_estimators=20, max_depth=12, random_state=seed, n_jobs=-1)
        t0 = time.time()
        rf.fit(X_train, y_train)
        rf_time = time.time() - t0
        rf_preds = rf.predict(X_test)
        rf_f1 = f1_score(y_test, rf_preds, average='macro')
        results_list.append({"Dataset": f"Synthetic_{n_samples}", "Seed": seed, "Model": "Scikit-Learn Exact RF", "Time_sec": rf_time, "F1_Macro": rf_f1})

        # --- 2. Extra-Trees ---
        et = ExtraTreesClassifier(n_estimators=20, max_depth=12, random_state=seed, n_jobs=-1)
        t0 = time.time()
        et.fit(X_train, y_train)
        et_time = time.time() - t0
        et_preds = et.predict(X_test)
        et_f1 = f1_score(y_test, et_preds, average='macro')
        results_list.append({"Dataset": f"Synthetic_{n_samples}", "Seed": seed, "Model": "Extra-Trees", "Time_sec": et_time, "F1_Macro": et_f1})

        # --- 3. LightGBM (GOSS) ---
        lgb = LGBMClassifier(n_estimators=20, max_depth=12, boosting_type='goss', random_state=seed, verbose=-1, n_jobs=-1)
        t0 = time.time()
        lgb.fit(X_train, y_train)
        lgb_time = time.time() - t0
        lgb_preds = lgb.predict(X_test)
        lgb_f1 = f1_score(y_test, lgb_preds, average='macro')
        results_list.append({"Dataset": f"Synthetic_{n_samples}", "Seed": seed, "Model": "LightGBM (GOSS)", "Time_sec": lgb_time, "F1_Macro": lgb_f1})

        # --- 4. Baseline FastForest MABSplit (ThrunGroup) ---
        try:
            ff_base = FastForestMAB(n_estimators=20, max_depth=12, solver='MAB')
            t0 = time.time()
            ff_base.fit(X_train, y_train)
            ff_base_time = time.time() - t0
            ff_base_preds = ff_base.predict(X_test)
            ff_base_f1 = f1_score(y_test, ff_base_preds, average='macro')
            results_list.append({"Dataset": f"Synthetic_{n_samples}", "Seed": seed, "Model": "Baseline FastForest MABSplit", "Time_sec": ff_base_time, "F1_Macro": ff_base_f1})
        except Exception as e:
            print(f"  [Baseline MABSplit     ] Seed {seed} | FAILED: {e}")

        # --- 5. OUR Accelerated FastMAB (FinalYear) ---
        try:
            our_mab = OurFastMAB(n_estimators=20, max_depth=12, n_jobs=-1, random_state=seed)
            t0 = time.time()
            our_mab.fit(X_train, y_train)
            our_mab_time = time.time() - t0
            our_mab_preds = our_mab.predict(X_test)
            our_mab_f1 = f1_score(y_test, our_mab_preds, average='macro')
            results_list.append({"Dataset": f"Synthetic_{n_samples}", "Seed": seed, "Model": "OUR FastMAB (FinalYear)", "Time_sec": our_mab_time, "F1_Macro": our_mab_f1})
        except Exception as e:
            print(f"  [OUR FastMAB (FinalYear)] Seed {seed} | FAILED: {e}")

df_res = pd.DataFrame(results_list)
summary = df_res.groupby(["Dataset", "Model"]).agg({"Time_sec": ["mean"], "F1_Macro": ["mean"]}).reset_index()
print("\n" + "="*85)
print("📊 FINAL SMALL DATASET BENCHMARK SUMMARY (Averaged across 3 seeds):")
print("="*85)
print(summary.to_string(index=False))